# Qwen3.5 Financial Analysis LoRA Training

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ramtin-asadi/Quantitative-Finance-Lab/blob/main/models/qwen_train_lora.ipynb)

In [ ]:
import os
import sys
import importlib.util
from pathlib import Path
from urllib.request import urlopen

on_colab = importlib.util.find_spec("google") is not None and importlib.util.find_spec("google.colab") is not None
use_drive = True
drive_project = Path("/content/drive/MyDrive/quantfinlab_training")
if on_colab:
    if use_drive:
        from google.colab import drive
        drive.mount("/content/drive")
    training_home = drive_project if use_drive else Path("/content/quantfinlab_training")
    model_dir = Path("/content/quantfinlab-models")
    model_dir.mkdir(exist_ok=True)
    for name in ["requirements-training.in", "requirements-training.lock", "requirements-export.in", "requirements-export.lock"]:
        bundled = training_home / name
        destination = model_dir / name
        if bundled.exists():
            destination.write_bytes(bundled.read_bytes())
        elif not destination.exists():
            url = "https://raw.githubusercontent.com/ramtin-asadi/Quantitative-Finance-Lab/main/models/" + name
            with urlopen(url, timeout=60) as response:
                destination.write_bytes(response.read())
    data_dir = training_home / "data"
    run_dir = training_home / "runs/qwen-financial-lora"
    cache_dir = training_home / "cache/huggingface"
else:
    model_dir = Path.cwd() if Path("requirements-training.in").exists() else Path.cwd() / "models"
    model_dir = model_dir.resolve()
    data_dir = model_dir / "data" if (model_dir / "data/manifest.json").exists() else model_dir.parent / "workspace/financial_analyst/training/frozen"
    run_dir = model_dir / "outputs/qwen-financial-lora"
    cache_dir = model_dir / "local/huggingface"
base_model = "Qwen/Qwen3.5-2B"
base_revision = "15852e8c16360a2fea060d615a32b45270f8a8fc"
llama_revision = "3057bb66c86c46d5781e50e85462a760ba7d1feb"
training_context = 8192
generation_tokens = 1536
generation_seconds = 240
seed = 3407
epochs = 2
learning_rate = 5e-5
batch_size = 1
gradient_accumulation = 8
evaluation_steps = 50
early_stopping_patience = 2
acceptance_per_task = 6
validate_gguf = True
lora_config = {"rank": 16, "alpha": 16, "dropout": 0.0, "bias": "none",
               "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]}
os.environ["HF_HOME"] = str(cache_dir)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
run_dir.mkdir(parents=True, exist_ok=True)
print("Training data:", data_dir, "\nPersistent output:", run_dir, "\nBase-model cache:", cache_dir)

In [ ]:
import importlib.metadata
import subprocess

if sys.platform != "linux" or sys.version_info[:2] not in {(3, 12), (3, 13)}:
    raise RuntimeError("Use Python 3.12 or 3.13 on Linux, Colab, or WSL2. See models/README.md for setup.")
requirements = model_dir / "requirements-training.in"
missing = []
for spec in requirements.read_text(encoding="utf-8").splitlines():
    if not spec.strip() or spec.startswith("--"):
        continue
    name, expected = spec.split("==")
    try:
        installed = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != expected:
        missing.append(spec)
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(requirements)], check=True)
    raise RuntimeError("Dependencies installed. Restart this notebook's kernel, then run from the first cell.")
from packaging.requirements import Requirement

for spec in requirements.read_text(encoding="utf-8").splitlines():
    if "==" not in spec:
        continue
    package = Requirement(spec)
    for dependency in importlib.metadata.requires(package.name) or []:
        dependency = Requirement(dependency)
        if dependency.marker and not dependency.marker.evaluate():
            continue
        if importlib.metadata.version(dependency.name) not in dependency.specifier:
            raise RuntimeError(f"Training dependency conflict: {package.name} requires {dependency}")
print("Pinned training environment is ready.")

In [ ]:
from unsloth import FastVisionModel

import gc
import hashlib
import json
import re
import shutil
import time
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from IPython.display import Markdown, display
from transformers import (AutoTokenizer, DataCollatorForSeq2Seq, EarlyStoppingCallback,
                          StoppingCriteria, StoppingCriteriaList, Trainer, TrainerCallback,
                          TrainingArguments, set_seed)
from transformers.trainer_utils import get_last_checkpoint

if torch.__version__.split("+")[0] != "2.8.0":
    raise RuntimeError("Restart the kernel to load the installed PyTorch version before training.")
if not torch.cuda.is_available():
    raise RuntimeError("Training needs an NVIDIA CUDA GPU. CPU-only conversion can be run separately on saved merged weights.")
gpu = torch.cuda.get_device_properties(0)
bf16 = torch.cuda.is_bf16_supported(including_emulation=False)
dtype = torch.bfloat16 if bf16 else torch.float16
if gpu.total_memory < 14 * 2**30:
    raise RuntimeError("This 16-bit LoRA recipe needs at least 15 GB VRAM. It does not load a 4-bit training base.")
if shutil.disk_usage(run_dir).free < 25 * 2**30:
    raise RuntimeError("Keep at least 25 GiB free for the cached base, checkpoints, merged weights and GGUF files.")
set_seed(seed)
display(pd.Series({"gpu": gpu.name, "vram_gib": gpu.total_memory / 2**30,
                   "base_precision": str(dtype), "adapter_precision": "float32"}))

In [ ]:
def log(message, **fields):
    stamp = datetime.now(timezone.utc).strftime("%H:%M:%S UTC")
    print(f"[{stamp}] {message}" + (" | " + ", ".join(f"{key}={value}" for key, value in fields.items()) if fields else ""), flush=True)

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def save_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding="utf-8")
    temporary.replace(path)

def run_visible(command):
    log("Running", command=" ".join(map(str, command)))
    with (run_dir / "export.log").open("a", encoding="utf-8") as stream:
        process = subprocess.Popen(list(map(str, command)), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, encoding="utf-8", errors="replace", bufsize=1)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                stream.write(line)
                stream.flush()
            returncode = process.wait()
        except BaseException:
            process.terminate()
            process.wait()
            raise
    if returncode:
        raise RuntimeError(f"Command failed with exit code {returncode}; see {run_dir / 'export.log'}.")

In [ ]:
manifest = json.loads((data_dir / "manifest.json").read_text(encoding="utf-8"))
if manifest["status"] != "frozen":
    raise ValueError("Use a reviewed, frozen train/validation split.")
records = {}
for split in ["train", "validation"]:
    path = data_dir / f"{split}.jsonl"
    if sha256(path) != manifest["files"][path.name]:
        raise ValueError(f"Dataset checksum differs from its manifest: {path.name}")
    records[split] = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    if len(records[split]) != manifest["counts"][split]:
        raise ValueError(f"Unexpected number of {split} examples.")
    for row in records[split]:
        packet = json.loads(row["messages"][1]["content"])
        cutoff = pd.Timestamp(packet["as_of"])
        if row["quality_status"] != "accepted" or cutoff != pd.Timestamp(row["cutoff"]):
            raise ValueError(f"Unreviewed example or inconsistent cutoff: {row['example_id']}")
        for evidence in packet["evidence"]:
            if pd.Timestamp(evidence["available_at"]) > cutoff or hashlib.sha256(evidence["text"].encode()).hexdigest() != evidence["text_hash"]:
                raise ValueError(f"Invalid evidence date or text hash: {row['example_id']}")
for field in ["example_id", "group_id", "source_ids", "source_hashes"]:
    inventories = []
    for split in ["train", "validation"]:
        values = set()
        for row in records[split]:
            value = row[field]
            values.update(value.values() if isinstance(value, dict) else value if isinstance(value, list) else [value])
        inventories.append(values)
    if inventories[0] & inventories[1]:
        raise ValueError(f"Training/validation leakage in {field}.")
train_end = max(pd.Timestamp(row["cutoff"]) for row in records["train"])
validation_start = min(pd.Timestamp(row["cutoff"]) for row in records["validation"])
if train_end >= validation_start:
    raise ValueError("Validation must be strictly later than training.")
task_counts = pd.DataFrame({split: pd.Series([row["task"] for row in rows]).value_counts()
                            for split, rows in records.items()})
display(task_counts)
task_counts.plot.bar(title="Training and later validation examples", ylabel="Examples", rot=20)
plt.show()
log("Dataset hashes and chronological separation passed", train_end=train_end, validation_start=validation_start)

In [ ]:
from typing import Literal
from pydantic import BaseModel, ConfigDict
from pydantic import Field, ValidationError
from decimal import Decimal

class Claim(BaseModel):
    model_config = ConfigDict(extra="forbid")
    statement: str = Field(min_length=1)
    evidence_ids: list[str] = Field(min_length=1)
    kind: Literal["fact", "interpretation", "uncertainty"]

class Analysis(BaseModel):
    model_config = ConfigDict(extra="forbid")
    conclusion: str = Field(min_length=1)
    materiality: Literal["low", "medium", "high", "uncertain"]
    claims: list[Claim] = Field(min_length=1)
    what_changed: str = Field(min_length=1)
    why_it_matters: str = Field(min_length=1)
    uncertainty: list[str] = Field(min_length=1)

def numerical_values(text):
    return {str(Decimal(value.replace(",", "")).normalize())
            for value in re.findall(r"(?<![\w])[-+]?\d[\d,]*(?:\.\d+)?", text)}

def check_output(text, row):
    try:
        result = Analysis.model_validate_json(text)
    except ValidationError as error:
        details = error.errors(include_url=False, include_input=False)
        return {"schema": False, "duplicate_claims": 0,
                "errors": [f"{item['loc']}: {item['msg']}" for item in details]}
    statements = [" ".join(claim.statement.casefold().split()) for claim in result.claims]
    duplicate_claims = len(statements) - len(set(statements))
    errors = ["Repeated claim statements"] if duplicate_claims else []
    evidence = {item["evidence_id"]: item for item in json.loads(row["messages"][1]["content"])["evidence"]}
    for i, claim in enumerate(result.claims):
        if set(claim.evidence_ids) - evidence.keys():
            errors.append(f"Claim {i}: unknown evidence ID")
            continue
        source = " ".join(evidence[key]["text"] for key in claim.evidence_ids)
        missing = numerical_values(claim.statement) - numerical_values(source)
        if missing:
            errors.append(f"Claim {i}: untraceable numbers {sorted(missing)}")
    prose = " ".join([result.conclusion, result.what_changed, result.why_it_matters, *result.uncertainty])
    missing = numerical_values(prose) - numerical_values(" ".join(e["text"] for e in evidence.values()))
    if missing:
        errors.append(f"Summary: untraceable numbers {sorted(missing)}")
    return {"schema": True, "duplicate_claims": duplicate_claims, "errors": errors}

def output_schema(row):
    schema = Analysis.model_json_schema()
    evidence = json.loads(row["messages"][1]["content"])["evidence"]
    schema["$defs"]["Claim"]["properties"]["evidence_ids"]["items"] = {
        "type": "string", "enum": [item["evidence_id"] for item in evidence]}
    return schema

for row in records["train"] + records["validation"]:
    result = check_output(row["messages"][-1]["content"], row)
    if not result["schema"] or result["errors"]:
        raise ValueError(f"Reference target failed validation: {row['example_id']}: {result['errors']}")


tasks = ["event", "sec_change", "macro", "reconciliation", "market"]
acceptance_rows = []
for task in tasks:
    group = sorted([row for row in records["validation"] if row["task"] == task], key=lambda row: row["example_id"])
    acceptance_rows.extend(group[:acceptance_per_task])
assert len(acceptance_rows) == len(tasks) * acceptance_per_task
print("Reference checks passed. Adapter checks:", len(acceptance_rows), "| GGUF checks:", len(tasks) if validate_gguf else 0)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(base_model, revision=base_revision, cache_dir=str(cache_dir))
tokenizer.padding_side = "right"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
chat_template = tokenizer.chat_template
chat_template_hash = hashlib.sha256(chat_template.encode()).hexdigest()

def prompt_text(messages):
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)

def encode_example(row):
    prompt = prompt_text(row["messages"][:2])
    prompt_ids = tokenizer.encode(prompt, add_special_tokens=False)
    completion_ids = tokenizer.encode(row["messages"][-1]["content"], add_special_tokens=False) + [tokenizer.eos_token_id]
    input_ids = prompt_ids + completion_ids
    if len(input_ids) > training_context:
        raise ValueError(f"Example exceeds the context; no silent truncation: {row['example_id']}")
    if not prompt.endswith("<|im_start|>assistant\n<think>\n\n</think>\n\n"):
        raise ValueError("Unexpected non-thinking assistant prefix; inspect the tokenizer template.")
    return {"input_ids": input_ids, "attention_mask": [1] * len(input_ids),
            "labels": [-100] * len(prompt_ids) + completion_ids}

encoded = {split: Dataset.from_list([encode_example(row) for row in rows]) for split, rows in records.items()}
lengths = pd.DataFrame([{"split": split, "task": row["task"], "tokens": len(item["input_ids"]),
                        "answer_tokens": sum(token != -100 for token in item["labels"])}
                       for split, rows in records.items() for row, item in zip(rows, encoded[split])])
display(lengths.groupby("split")[["tokens", "answer_tokens"]].describe().round(1))
lengths.groupby("split").tokens.plot.hist(alpha=0.5, bins=30, legend=True)
plt.xlabel("Prompt plus supervised answer tokens")
plt.show()

In [ ]:
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True, label_pad_token_id=-100,
                                pad_to_multiple_of=8, return_tensors="pt")
batch = collator([encoded["train"][0], encoded["train"][1]])
for row, original in zip(batch["labels"], [encoded["train"][0], encoded["train"][1]]):
    supervised = row[row.ne(-100)].tolist()
    expected = [token for token in original["labels"] if token != -100]
    assert supervised == expected and supervised[-1] == tokenizer.eos_token_id
    assert row[:len(original["labels"]) - len(expected)].eq(-100).all()
display(pd.DataFrame({"sequence_tokens": batch["attention_mask"].sum(1).tolist(),
                      "supervised_tokens": batch["labels"].ne(-100).sum(1).tolist()}))
display(Markdown("**Supervised answer, excluding EOS**\n\n" + tokenizer.decode(supervised[:-1])))
del batch
log("Completion-only masks, padding and EOS checked without a model forward pass")

In [ ]:
versions = {spec.split("==")[0]: importlib.metadata.version(spec.split("==")[0])
            for spec in requirements.read_text(encoding="utf-8").splitlines() if "==" in spec}
recipe = {"version": "qwen-financial-lora-1", "base_model": base_model, "base_revision": base_revision,
          "dataset_hashes": manifest["files"], "chat_template_hash": chat_template_hash, "lora": lora_config,
          "trainer": {"epochs": epochs, "learning_rate": learning_rate, "batch_size": batch_size,
                      "gradient_accumulation": gradient_accumulation, "training_context": training_context,
                      "dtype": str(dtype), "evaluation_steps": evaluation_steps,
                      "early_stopping_patience": early_stopping_patience, "seed": seed}, "packages": versions}
recipe_path = run_dir / "training_manifest.json"
if recipe_path.exists() and json.loads(recipe_path.read_text(encoding="utf-8")) != recipe:
    raise ValueError("This output folder belongs to another recipe. Choose a new run_dir to change the recipe.")
save_json(recipe_path, recipe)
checkpoints_dir = run_dir / "checkpoints"
checkpoints_dir.mkdir(exist_ok=True)
last_checkpoint = get_last_checkpoint(str(checkpoints_dir))
adapter_dir = run_dir / "adapter"
trained_path = run_dir / "trained.json"
trained = json.loads(trained_path.read_text(encoding="utf-8")) if trained_path.exists() else None
if trained and sha256(adapter_dir / "adapter_model.safetensors") != trained["adapter_sha256"]:
    raise ValueError("The completed adapter checksum changed.")
log("Training state", completed=trained is not None, resume_checkpoint=last_checkpoint)
log("Maximum scheduled updates", count=int(np.ceil(len(encoded["train"]) / (batch_size * gradient_accumulation)) * epochs))

In [ ]:
def load_model(adapter_path=None):
    log("Loading model", weights="saved adapter" if adapter_path else "pinned base")
    set_seed(seed)
    model, processor = FastVisionModel.from_pretrained(
        model_name=base_model, revision=base_revision, cache_dir=str(cache_dir),
        max_seq_length=training_context, load_in_4bit=False, dtype=dtype,
        use_gradient_checkpointing="unsloth", full_finetuning=False,
    )
    model = FastVisionModel.get_peft_model(
        model, finetune_vision_layers=False, finetune_language_layers=True,
        finetune_attention_modules=True, finetune_mlp_modules=True,
        target_modules=lora_config["target_modules"], r=lora_config["rank"],
        lora_alpha=lora_config["alpha"], lora_dropout=lora_config["dropout"],
        bias=lora_config["bias"], random_state=seed,
    )
    model.peft_config["default"].revision = base_revision
    model.peft_config["default"].base_model_name_or_path = base_model
    trainable = [(name, param) for name, param in model.named_parameters() if param.requires_grad]
    if not trainable or any("lora_" not in name for name, _ in trainable):
        raise ValueError("Only LoRA parameters should be trainable.")
    for name, param in trainable:
        if any(part in name.lower() for part in ["visual", "vision", "in_proj", "out_proj"]):
            raise ValueError(f"Unexpected trainable projection: {name}")
        if not any(f".{target}." in name for target in lora_config["target_modules"]):
            raise ValueError(f"Unexpected LoRA target: {name}")
        param.data = param.data.float()
    if adapter_path is not None:
        from peft import set_peft_model_state_dict
        from safetensors.torch import load_file
        weights = load_file(str(Path(adapter_path) / "adapter_model.safetensors"))
        loaded = set_peft_model_state_dict(model, weights, adapter_name="default")
        if loaded.unexpected_keys or any("lora_" in key for key in loaded.missing_keys):
            raise ValueError("Saved adapter weights do not match the configured LoRA modules.")
        model.requires_grad_(False)
    if getattr(model, "is_loaded_in_4bit", False):
        raise RuntimeError("This run requires a 16-bit base model.")
    for config in [model.config, model.config.text_config, model.generation_config]:
        config.eos_token_id = tokenizer.eos_token_id
        config.pad_token_id = tokenizer.pad_token_id
    log("Model loaded", lora_parameters=sum(param.numel() for _, param in trainable),
        base_precision=str(dtype), adapter_precision="float32", gpu_gib=round(torch.cuda.memory_allocated() / 2**30, 2))
    return model

model = load_model(adapter_dir if trained else None)
model.config.use_cache = False

In [ ]:
class FiniteLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        result = super().compute_loss(model, inputs, return_outputs=return_outputs,
                                      num_items_in_batch=num_items_in_batch)
        loss = result[0] if return_outputs else result
        if not torch.isfinite(loss.detach()).all().item():
            raise FloatingPointError("Non-finite loss; restart from the last saved optimizer checkpoint after inspecting the batch.")
        return result

class TrainingProgress(TrainerCallback):
    def __init__(self):
        self.accelerator = None
        self.started = time.monotonic()
        self.skipped_updates = 0
        self.consecutive_skips = 0

    def on_step_end(self, args, state, control, **kwargs):
        skipped = self.accelerator.optimizer_step_was_skipped
        self.skipped_updates += int(skipped)
        self.consecutive_skips = self.consecutive_skips + 1 if skipped else 0
        if skipped:
            log("AMP skipped an overflowed update and reduced its scale", step=state.global_step,
                scale=self.accelerator.scaler.get_scale())
        if self.consecutive_skips >= 8:
            raise FloatingPointError("Eight consecutive AMP skips; inspect numerical stability before resuming.")

    def on_log(self, args, state, control, logs=None, **kwargs):
        row = {"step": state.global_step, "elapsed_seconds": round(time.monotonic() - self.started, 1),
               "amp_skips_this_session": self.skipped_updates,
               "gpu_gib": round(torch.cuda.memory_allocated() / 2**30, 2), **(logs or {})}
        with (run_dir / "training_log.jsonl").open("a", encoding="utf-8") as stream:
            stream.write(json.dumps(row) + "\n")
        log("Training progress", **row)

    def on_save(self, args, state, control, **kwargs):
        log("Model, optimizer, scheduler, RNG and scaler checkpoint saved", step=state.global_step)

progress = TrainingProgress()
arguments = TrainingArguments(output_dir=str(checkpoints_dir), num_train_epochs=epochs,
    per_device_train_batch_size=batch_size, per_device_eval_batch_size=1,
    gradient_accumulation_steps=gradient_accumulation, learning_rate=learning_rate,
    warmup_ratio=0.1, weight_decay=0.01, max_grad_norm=1.0, optim="adamw_torch",
    fp16=not bf16, bf16=bf16, logging_steps=5, logging_first_step=True, logging_nan_inf_filter=False,
    eval_strategy="steps", eval_steps=evaluation_steps, prediction_loss_only=True,
    save_strategy="steps", save_steps=evaluation_steps, save_total_limit=3, save_only_model=False,
    load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
    restore_callback_states_from_checkpoint=True,
    seed=seed, data_seed=seed, dataloader_num_workers=0, report_to="none", remove_unused_columns=False)
callbacks = [progress]
if early_stopping_patience:
    callbacks.append(EarlyStoppingCallback(early_stopping_patience=early_stopping_patience,
                                          early_stopping_threshold=0.0001))
trainer = None
if trained is None:
    FastVisionModel.for_training(model)
    trainer = FiniteLossTrainer(model=model, args=arguments, train_dataset=encoded["train"],
        eval_dataset=encoded["validation"], data_collator=collator, processing_class=tokenizer, callbacks=callbacks)
    progress.accelerator = trainer.accelerator
    if not bf16 and last_checkpoint is None:
        trainer.accelerator.scaler = torch.amp.GradScaler("cuda", init_scale=1024)

In [ ]:
if trained is None:
    log("Starting the single training phase", resume=last_checkpoint, early_stopping_patience=early_stopping_patience)
    result = trainer.train(resume_from_checkpoint=last_checkpoint)
    if not np.isfinite(result.training_loss):
        raise FloatingPointError("Final training loss is not finite.")
    adapter_dir.mkdir(exist_ok=True)
    model.save_pretrained(str(adapter_dir), safe_serialization=True)
    tokenizer.save_pretrained(str(adapter_dir))
    trainer.state.save_to_json(str(run_dir / "trainer_state.json"))
    trained = {"adapter_sha256": sha256(adapter_dir / "adapter_model.safetensors"),
               "scheduled_updates": trainer.state.global_step, "training_loss": result.training_loss,
               "best_checkpoint": trainer.state.best_model_checkpoint, "best_eval_loss": trainer.state.best_metric,
               "amp_skips_this_session": progress.skipped_updates, "step_count_includes_amp_skips": True}
    save_json(trained_path, trained)
    log("Completed adapter saved", path=adapter_dir)
else:
    log("Completed adapter reused; training skipped")
display(pd.Series(trained))

In [ ]:
state = json.loads((run_dir / "trainer_state.json").read_text(encoding="utf-8"))
history = pd.DataFrame(state["log_history"])
fig, ax = plt.subplots(figsize=(9, 3.5))
for column in ["loss", "eval_loss"]:
    if column in history:
        history.dropna(subset=[column]).plot(x="step", y=column, marker=".", ax=ax)
ax.set(title="Completion-only training and validation loss", ylabel="Loss", xlabel="Scheduled update")
plt.show()
if trainer is not None:
    trainer.optimizer = None
    trainer.lr_scheduler = None
del trainer
progress.accelerator = None
gc.collect()
torch.cuda.empty_cache()
FastVisionModel.for_inference(model)
model.config.use_cache = True

In [ ]:
generation_settings = {"do_sample": False, "repetition_penalty": 1.0, "max_new_tokens": generation_tokens,
                       "max_time": generation_seconds, "check_version": "financial-checks-1"}

class GenerationProgress(StoppingCriteria):
    def __init__(self, prompt_length, label, budget):
        self.prompt_length = prompt_length
        self.label = label
        self.budget = budget
        self.started = time.monotonic()
        self.continue_generation = torch.zeros(1, dtype=torch.bool, device="cuda")

    def __call__(self, input_ids, scores, **kwargs):
        count = input_ids.shape[1] - self.prompt_length
        if count % 64 == 0:
            elapsed = time.monotonic() - self.started
            print(f"  {self.label}: {count}/{self.budget} tokens, {elapsed:.0f}s, {count / max(elapsed, 0.001):.1f} tokens/s", flush=True)
        return self.continue_generation

def decode_result(raw, row):
    output = raw.strip()
    fence = re.fullmatch(r"```(?:json)?\s*(\{.*\})\s*```", output, flags=re.S)
    if fence:
        output = fence.group(1).strip()
    return {"output": output, "raw_output": raw, **check_output(output, row)}

def generate(model, row, messages=None):
    messages = row["messages"][:2] if messages is None else messages
    inputs = tokenizer(prompt_text(messages), return_tensors="pt", add_special_tokens=False).to("cuda")
    prompt_length = inputs.input_ids.shape[1]
    budget = min(generation_tokens, training_context - prompt_length)
    if budget < 512:
        raise ValueError("The prompt leaves insufficient room for a complete answer.")
    meter = GenerationProgress(prompt_length, row["example_id"], budget)
    started = time.monotonic()
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=budget, max_time=generation_seconds,
            do_sample=False, repetition_penalty=1.0, use_cache=True,
            eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id,
            stopping_criteria=StoppingCriteriaList([meter]), return_dict_in_generate=False)
    answer = output[0, prompt_length:]
    stopped_eos = bool(len(answer) and answer[-1].item() == tokenizer.eos_token_id)
    elapsed = time.monotonic() - started
    result = {**decode_result(tokenizer.decode(answer, skip_special_tokens=True), row),
              "generated_tokens": len(answer), "stopped_eos": stopped_eos, "seconds": round(elapsed, 2)}
    if not stopped_eos:
        result["errors"].append("Generation reached its token or time budget before EOS")
    return result

def repair_messages(row, first):
    correction = ("Correct the answer using only the original evidence. Errors: " + json.dumps(first["errors"])
                  + ". Return a complete, concise JSON object. Remove unsupported numbers and repeated claims.")
    return [*row["messages"][:2], {"role": "assistant", "content": first["output"]},
            {"role": "user", "content": correction}]

In [ ]:
checks_path = run_dir / "adapter_checks.json"
check_identity = {"adapter_sha256": trained["adapter_sha256"], "validation_sha256": manifest["files"]["validation.jsonl"],
                  "settings": generation_settings, "examples": [row["example_id"] for row in acceptance_rows]}
saved = json.loads(checks_path.read_text(encoding="utf-8")) if checks_path.exists() else {}
acceptance = saved.get("results", []) if saved.get("identity") == check_identity else []
done = {item["example_id"] for item in acceptance}
for row in acceptance_rows:
    if row["example_id"] in done:
        continue
    log("Checking adapter", case=len(acceptance) + 1, total=len(acceptance_rows), task=row["task"])
    first = generate(model, row)
    attempts = [first]
    if first["errors"]:
        attempts.append(generate(model, row, repair_messages(row, first)))
    acceptance.append({"example_id": row["example_id"], "task": row["task"],
                       "attempts": attempts, **attempts[-1]})
    save_json(checks_path, {"identity": check_identity, "results": acceptance})
display(pd.DataFrame(acceptance)[["task", "schema", "stopped_eos", "duplicate_claims", "errors"]])
adapter_passed = all(not item["errors"] for item in acceptance)
log("Adapter checks complete", passed=adapter_passed, cases=len(acceptance))
if not adapter_passed:
    raise ValueError("Adapter checks failed. Saved weights and outputs remain available; inspect adapter_checks.json before export.")

In [ ]:
for task in tasks:
    item = next(item for item in acceptance if item["task"] == task)
    row = next(row for row in acceptance_rows if row["example_id"] == item["example_id"])
    answer = Analysis.model_validate_json(item["output"])
    display(Markdown("**" + task.replace("_", " ").title() + "**"))
    print(json.loads(row["messages"][1]["content"])["question"])
    print(" ".join(dict.fromkeys([answer.conclusion, answer.what_changed, answer.why_it_matters])))
    display(pd.DataFrame([claim.model_dump() for claim in answer.claims]))
    print("Uncertainty:", " ".join(answer.uncertainty))
print("Automatic checks do not establish financial meaning or causal correctness. Review these answers against their source packets.")

In [ ]:
merged_dir = run_dir / "merged"
merge_receipt_path = run_dir / "merge_receipt.json"
merge_identity = {"adapter_sha256": trained["adapter_sha256"], "base_revision": base_revision,
                  "chat_template_sha256": chat_template_hash}
saved_merge = json.loads(merge_receipt_path.read_text(encoding="utf-8")) if merge_receipt_path.exists() else {}
reuse_merge = saved_merge.get("identity") == merge_identity and all(
    (merged_dir / name).is_file() and sha256(merged_dir / name) == digest for name, digest in saved_merge.get("files", {}).items())
reuse_merge = reuse_merge and bool(saved_merge.get("files"))
if not reuse_merge:
    if merged_dir.exists() and any(merged_dir.iterdir()):
        raise ValueError("An incomplete or changed merge exists. Rename that folder before rebuilding it.")
    log("Merging LoRA into 16-bit weights", destination=merged_dir)
    model.save_pretrained_merged(str(merged_dir), tokenizer, save_method="merged_16bit")
    weights = list(merged_dir.glob("model*.safetensors"))
    if not weights or not (merged_dir / "config.json").is_file() or not (merged_dir / "tokenizer_config.json").is_file():
        raise ValueError("Merged weights or tokenizer files are incomplete.")
    files = {path.name: sha256(path) for path in merged_dir.iterdir() if path.is_file()}
    save_json(merge_receipt_path, {"identity": merge_identity, "files": files})
else:
    log("Verified merged weights reused")
del model
gc.collect()
torch.cuda.empty_cache()
log("Training model released; remaining export runs on CPU")

In [ ]:
tools_dir = model_dir / "local/export-tools"
tools_dir.mkdir(parents=True, exist_ok=True)
converter_dir = tools_dir / "python312"
converter_python = converter_dir / "bin/python"
converter_probe = "import sys, pip; assert sys.version_info[:2] == (3, 12)"
converter_ready = converter_python.exists() and subprocess.run(
    [str(converter_python), "-c", converter_probe], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0
if not converter_ready:
    run_visible([sys.executable, "-m", "pip", "install", "uv==0.8.22"])
    run_visible([sys.executable, "-m", "uv", "venv", "--python", "3.12", "--seed", "--clear", str(converter_dir)])
run_visible([converter_python, "-m", "pip", "install", "-r", model_dir / "requirements-export.in"])
run_visible([converter_python, "-m", "pip", "check"])
run_visible([converter_python, "-c", "import sys; from transformers import AutoTokenizer; t=AutoTokenizer.from_pretrained(sys.argv[1], local_files_only=True); print(type(t).__name__, len(t))", merged_dir])
llama_dir = tools_dir / "llama.cpp"
if not llama_dir.exists():
    run_visible(["git", "clone", "https://github.com/ggml-org/llama.cpp", llama_dir])
run_visible(["git", "-C", llama_dir, "checkout", llama_revision])
versions = subprocess.check_output([str(converter_python), "-m", "pip", "freeze"], text=True)
(run_dir / "converter_versions.txt").write_text(versions, encoding="utf-8")

In [ ]:
conversion_dir = run_dir / "conversion"
conversion_dir.mkdir(exist_ok=True)
fp16_gguf = conversion_dir / "qwen-financial-f16.gguf"
fp16_receipt = conversion_dir / "f16_receipt.json"
conversion_identity = {**merge_identity, "llama_revision": llama_revision}
saved = json.loads(fp16_receipt.read_text(encoding="utf-8")) if fp16_receipt.exists() else {}
reuse_fp16 = (fp16_gguf.is_file() and saved.get("identity") == conversion_identity
              and saved.get("sha256") == sha256(fp16_gguf))
if not reuse_fp16:
    partial = fp16_gguf.with_suffix(".gguf.partial")
    run_visible([converter_python, llama_dir / "convert_hf_to_gguf.py", merged_dir,
                 "--outfile", partial, "--outtype", "f16"])
    partial.replace(fp16_gguf)
    save_json(fp16_receipt, {"identity": conversion_identity, "sha256": sha256(fp16_gguf)})
else:
    log("Verified F16 conversion reused")

In [ ]:
build_dir = llama_dir / "build-cpu"
quantizer = build_dir / "bin/llama-quantize"
llama_server = build_dir / "bin/llama-server"
build_receipt = tools_dir / "cpu_build.json"
build_identity = {"revision": llama_revision, "cuda": False, "server": validate_gguf}
saved = json.loads(build_receipt.read_text(encoding="utf-8")) if build_receipt.exists() else {}
if saved != build_identity or not quantizer.is_file() or (validate_gguf and not llama_server.is_file()):
    log("Building CPU export tools", workers=min(4, os.cpu_count() or 2))
    run_visible(["cmake", "-S", llama_dir, "-B", build_dir, "-DCMAKE_BUILD_TYPE=Release",
                 "-DGGML_CUDA=OFF", "-DLLAMA_CURL=OFF", "-DLLAMA_BUILD_TESTS=OFF"])
    targets = ["llama-quantize", "llama-server"] if validate_gguf else ["llama-quantize"]
    run_visible(["cmake", "--build", build_dir, "--config", "Release", "-j", str(min(4, os.cpu_count() or 2)), "--target", *targets])
    save_json(build_receipt, build_identity)
else:
    log("CPU export tools reused")

In [ ]:
export_dir = run_dir / "export"
export_dir.mkdir(exist_ok=True)
gguf_path = export_dir / "qwen-financial-Q4_K_M.gguf"
quantization_receipt = export_dir / "quantization.json"
saved = json.loads(quantization_receipt.read_text(encoding="utf-8")) if quantization_receipt.exists() else {}
reuse_quantization = (gguf_path.is_file() and saved.get("identity") == conversion_identity
                      and saved.get("sha256") == sha256(gguf_path))
if not reuse_quantization:
    partial = gguf_path.with_suffix(".gguf.partial")
    run_visible([quantizer, fp16_gguf, partial, "Q4_K_M"])
    partial.replace(gguf_path)
    save_json(quantization_receipt, {"identity": conversion_identity, "sha256": sha256(gguf_path)})
else:
    log("Verified Q4_K_M quantization reused")
with gguf_path.open("rb") as stream:
    if stream.read(4) != b"GGUF":
        raise ValueError("The output is not a GGUF file.")
model_sha = sha256(gguf_path)
log("Local model exported", path=gguf_path, gib=round(gguf_path.stat().st_size / 2**30, 3))

In [ ]:
import socket

import requests

gguf_rows = [next(row for row in acceptance_rows if row["task"] == task) for task in tasks]
gguf_checks_path = run_dir / "gguf_checks.json"
gguf_identity = {"sha256": model_sha, "llama_revision": llama_revision, "context": training_context,
                 "generation_tokens": generation_tokens, "examples": [row["example_id"] for row in gguf_rows]}
saved = json.loads(gguf_checks_path.read_text(encoding="utf-8")) if gguf_checks_path.exists() else {}
gguf_checks = saved.get("results", []) if saved.get("identity") == gguf_identity else []
done = {item["example_id"] for item in gguf_checks}
if validate_gguf and len(done) < len(gguf_rows):
    with socket.socket() as socket_probe:
        socket_probe.bind(("127.0.0.1", 0))
        port = socket_probe.getsockname()[1]
    endpoint = f"http://127.0.0.1:{port}"
    with (run_dir / "gguf_server.log").open("w", encoding="utf-8") as server_log:
        server = subprocess.Popen([str(llama_server), "-m", str(gguf_path), "-c", str(training_context),
            "-ngl", "0", "--parallel", "1", "--host", "127.0.0.1", "--port", str(port),
            "--jinja", "--reasoning-format", "none"], stdout=server_log, stderr=subprocess.STDOUT)
        try:
            started = time.monotonic()
            while True:
                if server.poll() is not None:
                    raise RuntimeError("GGUF server exited; inspect gguf_server.log.")
                try:
                    ready = requests.get(endpoint + "/health", timeout=2).ok
                except requests.RequestException:
                    ready = False
                if ready:
                    break
                if time.monotonic() - started > 180:
                    raise TimeoutError("GGUF server did not become ready; inspect gguf_server.log.")
                time.sleep(1)
            for row in gguf_rows:
                if row["example_id"] in done:
                    continue
                messages, attempts = row["messages"][:2], []
                for attempt in range(2):
                    log("Checking quantized model on CPU", task=row["task"], attempt=attempt + 1)
                    request = {"messages": messages, "temperature": 0, "seed": seed, "max_tokens": generation_tokens,
                               "chat_template_kwargs": {"enable_thinking": False},
                               "response_format": {"type": "json_schema", "json_schema": {"name": "analysis", "schema": output_schema(row)}}}
                    response = requests.post(endpoint + "/v1/chat/completions", json=request, timeout=600)
                    response.raise_for_status()
                    choice = response.json()["choices"][0]
                    result = decode_result(choice["message"]["content"], row)
                    if choice["finish_reason"] != "stop":
                        result["errors"].append("GGUF generation did not end normally")
                    attempts.append(result)
                    if not result["errors"]:
                        break
                    messages = repair_messages(row, result)
                gguf_checks.append({"example_id": row["example_id"], "task": row["task"], "attempts": attempts, **attempts[-1]})
                save_json(gguf_checks_path, {"identity": gguf_identity, "results": gguf_checks})
        finally:
            server.terminate()
            try:
                server.wait(timeout=20)
            except subprocess.TimeoutExpired:
                server.kill()
                server.wait()
if validate_gguf:
    display(pd.DataFrame(gguf_checks)[["task", "schema", "errors"]])
    if any(item["errors"] for item in gguf_checks):
        raise ValueError("Quantized checks failed. The model and full outputs are saved; inspect gguf_checks.json.")

In [ ]:
export_manifest = {"base_model": base_model, "base_revision": base_revision, "adapter_sha256": trained["adapter_sha256"],
                   "filename": gguf_path.name, "sha256": model_sha, "quantization": "Q4_K_M",
                   "training_context": training_context, "llama_cpp_revision": llama_revision,
                   "chat_template_sha256": chat_template_hash, "enable_thinking": False,
                   "adapter_checks_passed": adapter_passed, "gguf_generation_checked": validate_gguf,
                   "gguf_checks_passed": all(not item["errors"] for item in gguf_checks) if validate_gguf else None}
save_json(export_dir / "export_manifest.json", export_manifest)
save_json(export_dir / "output_schema.json", Analysis.model_json_schema())
save_json(export_dir / "dataset_manifest.json", manifest)
(export_dir / "SHA256SUMS").write_text(f"{model_sha}  {gguf_path.name}\n", encoding="utf-8")
(export_dir / "chat_template.jinja").write_text(chat_template, encoding="utf-8")
display(pd.DataFrame([{"artifact": label, "path": str(path)} for label, path in
                     [("LoRA adapter", adapter_dir), ("merged weights", merged_dir), ("Q4_K_M model", gguf_path),
                      ("training log", run_dir / "training_log.jsonl"), ("adapter checks", checks_path),
                      ("export manifest", export_dir / "export_manifest.json")]]))
log("Training and local export complete")
print("Keep the adapter, manifests and quantized model. Checkpoints permit training resumption; the cached base and merge avoid repeat downloads and conversion.")